# diffBloch Event Report

Renders summary tables and figures from the canonical `ReportLogger` JSONL event stream. The
refinement library writes no images; plotting and optional figure export happen here.

The tables live in `tools/event_report/tables.py`, the plotting code in
`tools/event_report/figures.py`, and the loading code in `tools/event_report/reader.py` —
importable modules with test coverage, not notebook cells. This notebook is the viewer over them.

Point it at a report by setting `REPORT` in the cell below, or by launching Jupyter with
`DIFFBLOCH_EVENT_LOG=/path/to/report.jsonl`. Left alone, it opens `example_report.jsonl` beside
this notebook: a real `diffbloch refine` of the bundled quartz-no-abs example.

In [ ]:
# Setup: locate the checkout and the report to render.
import sys
from pathlib import Path

from IPython.display import Markdown, display

try:
    from tools.event_report import figures, reader, tables
except ModuleNotFoundError:
    # Launched from somewhere other than the checkout root: find it and retry.
    for _candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (_candidate / "pyproject.toml").is_file() and (
            _candidate / "src" / "diffBloch"
        ).is_dir():
            sys.path.insert(0, str(_candidate))
            break
    from tools.event_report import figures, reader, tables

# The report to render. Edit this, or set DIFFBLOCH_EVENT_LOG before launching Jupyter.
REPORT = reader.default_event_log()
EXPORT_DIR = Path("event_report_figures")
EXPORT_FORMATS = ("svg",)

print(f"REPORT = {REPORT}")

In [ ]:
# Re-run this cell after changing REPORT.
records = reader.read_records(REPORT)
summary = tables.build_tables(records)
sections = figures.build_sections(records)
print(
    f"{len(records)} records -> {len(summary)} tables, {sum(len(f) for _, f in sections)} figures"
)

for title, table in summary:
    display(Markdown(f"## {title}\n\n{table}"))

for title, panels in sections:
    display(Markdown(f"## {title}"))
    for figure in panels.values():
        display(figure)

built = {name: figure for _, panels in sections for name, figure in panels.items()}

In [ ]:
# Figure export is opt-in and lives only here, never in the refinement library.
EXPORT_FIGURES = False
figures.export_figures(built, EXPORT_DIR, EXPORT_FORMATS) if EXPORT_FIGURES else []